# 4.17 Dictionary Learning & Sparse Coding

Dictionary learning represents each example with a small number of active atoms instead of dense coordinates.

Part 4 moves from prediction with labels to discovery without labels. Vectors, covariance, kernels, and matrix factorization become the language for hidden low-dimensional structure. Geometry supplies distances, neighborhoods, projections, and matrix shapes; optimization decides which structure is kept.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import FactorAnalysis
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Lasso
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def scale_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def pad_to_2d(Z):
    if Z.shape[1] >= 2:
        return Z[:, :2]
    zeros = np.zeros((Z.shape[0], 1))
    return np.hstack([Z, zeros])


def reconstruction_rmse(X, X_hat):
    return float(np.sqrt(np.mean((X - X_hat) ** 2)))


def plot_embedding_panels(results, metric_label):
    fig = plt.figure(figsize=(15, 6))
    for idx, item in enumerate(results):
        ax = fig.add_subplot(2, 5, idx + 1)
        Z2 = pad_to_2d(item["Z"])
        ax.scatter(Z2[:, 0], Z2[:, 1], c=item["y"], s=8, cmap="viridis", alpha=0.8)
        ax.set_title(item["name"].split("(")[0].strip(), fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    ax = fig.add_subplot(2, 1, 2)
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels(["D" + str(i) for i in xs])
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label + " vs ladder rung")
    fig.tight_layout()
    return fig


## The concept, built once: sparse codes after the linear audit

The lesson connects sparse coding to the same shape bookkeeping $$X_c=U\Sigma V^\top,\qquad Z=X_cV_r$$, then adds an $\ell_1$ penalty so most coefficients are zero.
We first verify the lesson means $[2.50,2.50]$, singular values $[2.828,1.414]$, and baseline energy $0.800$.

In [ ]:

toy = np.array([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
    [4.0, 3.0],
])

centered = toy - toy.mean(axis=0)
U, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)
energies = singular_values ** 2
first_share = energies[0] / energies.sum()

assert np.allclose(toy.mean(axis=0), [2.5, 2.5])
assert np.allclose(np.round(singular_values, 3), [2.828, 1.414])
assert np.isclose(np.round(first_share, 3), 0.800)

print("means", toy.mean(axis=0))
print("singular values", np.round(singular_values, 3))
print("first-component variance share", np.round(first_share, 3))


Now implement a real alternating sparse-coding routine: initialize atoms with PCA directions, solve Lasso codes for each example, update the dictionary by least squares, normalize atoms, and repeat.

In [ ]:

def normalize_dictionary(D):
    norms = np.linalg.norm(D, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return D / norms


def sparse_encode_lasso(Xc, D, alpha):
    codes = []
    design = D.T
    for row in Xc:
        model = Lasso(alpha=alpha, fit_intercept=False, max_iter=600)
        model.fit(design, row)
        codes.append(model.coef_)
    return np.vstack(codes)


def dictionary_learning_method(X, n_atoms=4, alpha=0.05, n_iter=8, random_state=7):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    Xc = X - mean
    rng = np.random.default_rng(random_state)
    U, singular_values, Vt = np.linalg.svd(Xc, full_matrices=False)
    D = np.zeros((n_atoms, X.shape[1]))
    usable = min(n_atoms, Vt.shape[0])
    D[:usable] = Vt[:usable]
    if n_atoms > usable:
        D[usable:] = rng.normal(size=(n_atoms - usable, X.shape[1]))
    D = normalize_dictionary(D)
    for step in range(n_iter):
        codes = sparse_encode_lasso(Xc, D, alpha)
        if np.allclose(codes, 0.0):
            break
        D = np.linalg.lstsq(codes, Xc, rcond=None)[0]
        D = normalize_dictionary(D)
    codes = sparse_encode_lasso(Xc, D, alpha)
    X_hat = codes @ D + mean
    sparsity = float(np.mean(np.abs(codes) < 1e-8))
    return codes, X_hat, D, sparsity

codes_toy, X_hat_toy, D_toy, sparsity_toy = dictionary_learning_method(toy, n_atoms=2, alpha=0.01, n_iter=4)
assert codes_toy.shape == (4, 2)
assert D_toy.shape == (2, 2)
assert np.isclose(np.round(first_share, 3), 0.800)
print("sparse code shape", codes_toy.shape)
print("toy sparsity", np.round(sparsity_toy, 3))


## The dataset ladder

The same sparse dictionary method is applied to D1-D5. Two code coordinates are plotted; reconstruction error is the metric.

In [ ]:

rungs = dimred_ladder()
for idx, (name, X, y) in enumerate(rungs, start=1):
    y_array = np.asarray(y)
    unique_preview = np.unique(y_array[: min(len(y_array), 200)])[:8]
    print(f"D{idx}: {name}")
    print(f"  X shape: {X.shape}")
    print(f"  y preview: {unique_preview}")
    print(f"  first row: {np.round(X[0, : min(6, X.shape[1])], 3)}")


## Run the same dictionary-learning method across D1-D5

In [ ]:

results = []
for name, X, y in dimred_ladder():
    X_scaled = scale_for_geometry(X)
    codes, X_hat, D, sparsity = dictionary_learning_method(X_scaled, n_atoms=6, alpha=0.04, n_iter=6, random_state=7)
    rmse = reconstruction_rmse(X_scaled, X_hat)
    Z = codes[:, :2]
    results.append({"name": name, "Z": Z, "y": y, "metric": rmse, "sparsity": sparsity})

print("rung | reconstruction rmse | sparsity")
for idx, item in enumerate(results, start=1):
    print(f"D{idx} | {item['metric']:.3f} | {item['sparsity']:.3f} | {item['name']}")


## Results visualization

Panels show the first two sparse-code coordinates; the curve shows reconstruction error as data complexity increases.

In [ ]:

fig = plot_embedding_panels(results, "reconstruction rmse")
plt.show()


## Pitfall on D5: too many atoms can memorize noise

A lower training objective can be a worse model. Compare training and held-out reconstruction while sweeping atom count and sparsity.

In [ ]:

name, X_d5, y_d5 = dimred_ladder()[-1]
X_d5_scaled = scale_for_geometry(X_d5)
X_train, X_test = train_test_split(X_d5_scaled, test_size=0.25, random_state=7)
settings = [
    (4, 0.08),
    (8, 0.05),
    (16, 0.02),
]
for n_atoms, alpha in settings:
    codes_train, X_hat_train, D, sparsity = dictionary_learning_method(X_train, n_atoms=n_atoms, alpha=alpha, n_iter=6, random_state=7)
    codes_test = sparse_encode_lasso(X_test - X_train.mean(axis=0), D, alpha)
    X_hat_test = codes_test @ D + X_train.mean(axis=0)
    train_rmse = reconstruction_rmse(X_train, X_hat_train)
    test_rmse = reconstruction_rmse(X_test, X_hat_test)
    print(f"atoms={n_atoms} alpha={alpha} train={train_rmse:.3f} heldout={test_rmse:.3f} sparsity={sparsity:.3f}")


## Evaluate it + Practice

- Track the ladder metric: reconstruction RMSE, with held-out RMSE for overfitting checks. Compare it with a no-skill baseline such as using the first two raw scaled columns or the input mean reconstruction.
- Sanity check shapes: D1-D5 should all return a two-column visualization embedding and a scalar metric.
- Ablation: increase atoms and lower sparsity until training error improves but held-out error stalls or worsens; the metric should get worse or the visualization should lose structure.
- Failure signals: tiny hyperparameter changes flip the pattern, D5 improves only on the training objective, or one raw-scale feature dominates.
- Stability check: rerun with a nearby seed or hyperparameter and compare reconstruction, trustworthiness, or neighbor preservation rather than component names.

Practice 1: Change the number of components or atoms and redraw the metric curve.

Practice 2: Sweep alpha at a fixed atom count and plot sparsity versus RMSE.

Practice 3: On D3 digits, reshape several learned atoms into 8 by 8 images.